In [7]:
import yaml
import pandas as pd
from sqlalchemy import create_engine
from sqlalchemy import text


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)

# Leer credenciales del arhivo YAML
with open("credenciales.yaml", "r") as file:
     config = yaml.safe_load(file)

db = config["mysql"]

# Crear conexión
# Su función es crear un Engine, que es el objeto que SQLAlchemy utiliza para comunicarse con la base de datos.
engine = create_engine(                                     
    f"mysql+pymysql://{db['user']}:{db['password']}@"
    f"{db['host']}:{db['port']}/{db['database']}"
)

##### 4.- ANALISIS DE COHORTE

##### 4-1 Introduccion a las Cohortes

<span style="font-size: 12px;">

Agrupamiento generacional: Cual es caracteristica para agrupar gente en la cohorte, por ejemplo: fecha de nacimiento, primer compra de un producto, nacionalidad, género, sexo. No debe cambiar a lo largo del estudio.

Temporalidad: Tiempo en que se estará siguiendo al grupo.

Métrica agregada: Cómo mediremos a las personas de la cohorte, pueden ser ventas, ticket promedio, calificaciones, asistencia a clase, tasa de clicks, etc.


Temas:

Análisis de retención: Si cada miembro de una generación está presenta en una fecha en particular de nuestro estudio.

Análisis de superviviencia: Cuanta gente de una generación sobrevive después de cierto tiempo.

Análisis de retorno: Si una acción ha sucedido más de n veces en un periodo de tiempo dado. Por ejemplo, cuantas veces nos compra un cliente en un periodo de tiempo.

Análisis Acumulativos: A lo largo de la vida del cliente, cuento dinero ha gastado con nosotros.

</span>


##### 4-2 Introducción a Análisis de Retencion

<span style="font-size: 12px;">

Qué tanta gente se queda con nosotros. Ver el tamaño de la cohorte a través del tiempo, o bien, que tanto dinero gasta la gente a traves del tiempo. 

Buscamos expresar como un %. Al inicio se tiene un 100%, luego puede aumentar o disminuir.

Se puede considerar el número de personas o gasto de dinero a través del tiempo.

</span>

##### 4-3 SQL para curva basica de Retencion

In [8]:
query="""
SELECT
     * 
FROM
     cohort.supertienda
LIMIT
     10;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,venta_id,Categoria,Ciudad,Pais/Región,Nombre_del_Cliente,Proveedor,Fecha_de_Orden,ID_Orden,Codigo_Postal,Nombre_de_Producto,Region,Segmento,Fecha_de_envio,Estado/Provincia,Descuento,Ganancia,Cantidad,Ventas
0,1,Suministros de Oficina,Houston,United States,Darren Powers,Message Book,2019-01-03,US-2019-103800,77095,"Message Book, Wirebound, Four 5 1/2"" X 4"" Form...",Central,Consumidor,2019-01-07,Texas,0.2,5.5512,2,16.448
1,2,Suministros de Oficina,Naperville,United States,Phillina Ober,GBC,2019-01-04,US-2019-112326,60540,GBC Standard Plastic Binding Systems Combs,Central,Home Office,2019-01-08,Illinois,0.8,-5.4870,2,3.540
2,3,Suministros de Oficina,Naperville,United States,Phillina Ober,Avery,2019-01-04,US-2019-112326,60540,Avery 508,Central,Home Office,2019-01-08,Illinois,0.2,4.2717,3,11.784
3,4,Suministros de Oficina,Naperville,United States,Phillina Ober,SAFCO,2019-01-04,US-2019-112326,60540,SAFCO Boltless Steel Shelving,Central,Home Office,2019-01-08,Illinois,0.2,-64.7748,3,272.736
4,5,Suministros de Oficina,Philadelphia,United States,Mick Brown,Avery,2019-01-05,US-2019-141817,19143,Avery Hi-Liter EverBold Pen Style Fluorescent ...,Este,Consumidor,2019-01-12,Pennsylvania,0.2,4.8840,3,19.536
5,6,Muebles,Henderson,United States,Maria Etezadi,Global,2019-01-06,US-2019-167199,42420,Global Deluxe High-Back Manager's Chair,Sur,Home Office,2019-01-10,Kentucky,0.0,746.4078,9,2573.820
6,7,Suministros de Oficina,Henderson,United States,Maria Etezadi,Rogers,2019-01-06,US-2019-167199,42420,Rogers Handheld Barrel Pencil Sharpener,Sur,Home Office,2019-01-10,Kentucky,0.0,1.4796,2,5.480
7,8,Suministros de Oficina,Athens,United States,Jack O'Briant,Dixon,2019-01-06,US-2019-106054,30605,"Dixon Prang Watercolor Pencils, 10-Color Set w...",Sur,Corporativo,2019-01-07,Georgia,0.0,5.2398,3,12.780
8,9,Suministros de Oficina,Henderson,United States,Maria Etezadi,Ibico,2019-01-06,US-2019-167199,42420,Ibico Hi-Tech Manual Binding System,Sur,Home Office,2019-01-10,Kentucky,0.0,274.4910,2,609.980
9,10,Suministros de Oficina,Henderson,United States,Maria Etezadi,Alliance,2019-01-06,US-2019-167199,42420,"Alliance Super-Size Bands, Assorted Sizes",Sur,Home Office,2019-01-10,Kentucky,0.0,0.3112,4,31.120


In [9]:
# Convertir las columnas Fecha_deOrden y Fecha_de_envio a tipo date

query="""
ALTER TABLE cohort.supertienda
MODIFY COLUMN Fecha_de_Orden DATE,
MODIFY COLUMN Fecha_de_envio DATE;
"""
with engine.begin() as conn:
    resultado = conn.execute(text(query))

print(f"Filas actualizadas: {resultado.rowcount}")

Filas actualizadas: 0


In [10]:
# Obtener la fecha en que cada cliente realizó su primera orden

query="""
select
    nombre_del_cliente,
    min(fecha_de_orden) as fecha_primera_orden
from
    cohort.supertienda
group by
    1;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,nombre_del_cliente,fecha_primera_orden
0,Darren Powers,2019-01-03
1,Phillina Ober,2019-01-04
2,Mick Brown,2019-01-05
3,Maria Etezadi,2019-01-06
4,Jack O'Briant,2019-01-06
5,Lycoris Saunders,2019-01-06
6,Vivek Sundaresam,2019-01-07
7,Melanie Seite,2019-01-09
8,Anthony Jacobs,2019-01-10
9,Seth Vernon,2019-01-11


In [11]:
# Obtener cuantos clientes compraron, cuantos compraron 1 mes después de su primera compra, 2 meses después de su primera compra, etc

query="""
select
    timestampdiff(MONTH, A.fecha_primera_orden, B.fecha_de_orden) as periodo,
    count(distinct B.nombre_del_cliente) as nro_clientes
from
    (
    select
        nombre_del_cliente,
        min(fecha_de_orden) as fecha_primera_orden
    from
        cohort.supertienda
    group by 1
    ) AS A
    inner join
    cohort.supertienda as B on A.nombre_del_cliente = B.nombre_del_cliente
group by
    1;
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,periodo,nro_clientes
0,0,800
1,1,69
2,2,74
3,3,71
4,4,61
5,5,89
6,6,74
7,7,87
8,8,81
9,9,75


In [12]:
# Con la consulta anterior, obtener el porcentaje de cada periodo respecto al periodo 0

query="""
select
    periodo,
    nro_clientes,
    first_value(nro_clientes) over (order by periodo) as clientes_iniciales,
    (100 * nro_clientes / first_value(nro_clientes) over (order by periodo) ) as porcentaje_retencion
from
    (
    select
        timestampdiff(MONTH, A.fecha_primera_orden, B.fecha_de_orden) as periodo,
        count(distinct B.nombre_del_cliente ) as nro_clientes
    from 
        (
        select
            nombre_del_cliente,
            min(fecha_de_orden) as fecha_primera_orden
        from
            cohort.supertienda
        group by 	1
        ) AS A 
        inner join 
        cohort.supertienda as B on A.nombre_del_cliente = B.nombre_del_cliente
    group by
        1
    ) AS Z;
"""
df = pd.read_sql(text(query),engine)
df.head(10)


,periodo,nro_clientes,clientes_iniciales,porcentaje_retencion
0,0,800,800,100.000
1,1,69,800,8.625
2,2,74,800,9.250
3,3,71,800,8.875
4,4,61,800,7.625
5,5,89,800,11.125
6,6,74,800,9.250
7,7,87,800,10.875
8,8,81,800,10.125
9,9,75,800,9.375


##### 4-4 Analisis de Retencion Legisladores


In [13]:
query = """
SELECT
    * 
FROM
    cohort.legisladoresusa
LIMIT
    5;
"""
df = pd.read_sql(text(query),engine)
df.head()

,IDLegislador,DipSen,InicioPeriodo,FinPeriodo,Estado
0,B000944,rep,1993-01-05,1995-01-03,OH
1,C000127,rep,1993-01-05,1995-01-03,WA
2,C000141,rep,1987-01-06,1989-01-03,MD
3,C000174,rep,1983-01-03,1985-01-03,DE
4,C001070,sen,2007-01-04,2013-01-03,PA


In [14]:
# Explorar el contenido de la tabla de fechas

query="""
SELECT
    * 
FROM
    cohort.tablafechas
"""
df = pd.read_sql(text(query),engine)
df.head(10)


,Year,Mes,Dia,Dias en Year,Fecha
0,2020,12,31,366,2020-12-31
1,2019,12,31,365,2019-12-31
2,2018,12,31,365,2018-12-31
3,2017,12,31,365,2017-12-31
4,2016,12,31,366,2016-12-31
5,2015,12,31,365,2015-12-31
6,2014,12,31,365,2014-12-31
7,2013,12,31,365,2013-12-31
8,2012,12,31,366,2012-12-31
9,2011,12,31,365,2011-12-31


In [17]:
# Agregar una columna de fecha 
query_alter="""
ALTER TABLE cohort.tablafechas ADD COLUMN Fecha DATE;
"""

# Definir el contenido de la columna Fecha
query_update="""
UPDATE cohort.tablafechas SET cohort.tablafechas.Fecha = concat(Year,"-",Mes,"-",Dia);
"""

# Ejecutar las consultas de alteración y actualización de la tabla
with engine.begin() as conn:
    conn.execute(text(query_alter))                  # Ejecutar el query de alteración de la tabla
    resultado = conn.execute(text(query_update))     # Ejecutar el query de actualización de la tabla, para saber cuantas filas fueron actualizadas, se puede usar resultado.rowcount

print(f"Filas actualizadas: {resultado.rowcount}")

Filas actualizadas: 251


In [16]:
# En caso de necesitar eliminar la columna Fecha, se puede ejecutar la siguiente consulta:

query = """
ALTER TABLE cohort.tablafechas
DROP COLUMN Fecha;
"""

with engine.begin() as conn:
    conn.execute(text(query))

In [18]:
# Explorar el contenido actualizado de la tabla de fechas

query="""
SELECT
    * 
FROM
    cohort.tablafechas
"""
df = pd.read_sql(text(query),engine)
df.head(10)

,Year,Mes,Dia,Dias en Year,Fecha
0,2020,12,31,366,2020-12-31
1,2019,12,31,365,2019-12-31
2,2018,12,31,365,2018-12-31
3,2017,12,31,365,2017-12-31
4,2016,12,31,366,2016-12-31
5,2015,12,31,365,2015-12-31
6,2014,12,31,365,2014-12-31
7,2013,12,31,365,2013-12-31
8,2012,12,31,366,2012-12-31
9,2011,12,31,365,2011-12-31


In [19]:
# Cuando inicia cada legislador su periodo

query="""
SELECT
    IDLegislador,
    MIN(InicioPeriodo) as Primer_Periodo
FROM
    cohort.legisladoresusa
GROUP BY
    1
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,IDLegislador,Primer_Periodo
0,B000944,1993-01-05
1,C000127,1993-01-05
2,C000141,1987-01-06
3,C000174,1983-01-03
4,C001070,2007-01-04
5,F000062,1992-11-10
6,F000469,2019-01-03
7,K000367,2007-01-04
8,M000639,1993-01-05
9,S000033,1991-01-03


In [20]:
# Agregar la fecha de primer periodo a la tabla original de legisladores

query="""
SELECT
    *
FROM
    (
    SELECT
        IDLegislador,
        MIN(InicioPeriodo) as Primer_Periodo
    FROM
        cohort.legisladoresusa
    GROUP BY
        1
    ) AS A 
    INNER JOIN
    cohort.legisladoresusa AS B ON A.IDLegislador=B.IDLegislador;
"""
df = pd.read_sql(text(query), engine)
df.head()

,IDLegislador,Primer_Periodo,IDLegislador,DipSen,InicioPeriodo,FinPeriodo,Estado
0,B000944,1993-01-05,B000944,rep,1993-01-05,1995-01-03,OH
1,C000127,1993-01-05,C000127,rep,1993-01-05,1995-01-03,WA
2,C000141,1987-01-06,C000141,rep,1987-01-06,1989-01-03,MD
3,C000174,1983-01-03,C000174,rep,1983-01-03,1985-01-03,DE
4,C001070,2007-01-04,C001070,sen,2007-01-04,2013-01-03,PA


In [21]:
# Usar la tabla de fechas para agregar las fechas intermedias entre InicioPeriodo y FinPeriodo y calcular el periodo.

query="""
SELECT
    A.IDLegislador,
    A.Primer_Periodo,
    B.InicioPeriodo,
    B.FinPeriodo,
    C.Fecha,
    COALESCE(timestampdiff(YEAR,A.primer_periodo,C.fecha),0) as Periodo
FROM
    (
    SELECT
        IDLegislador,
        MIN(InicioPeriodo) as Primer_Periodo
    FROM
        cohort.legisladoresusa
    GROUP BY
        1
    ) AS A 
    INNER JOIN 
    cohort.legisladoresusa AS B ON A.IDLegislador=B.IDLegislador
    LEFT JOIN 
    cohort.tablafechas AS C ON C.FECHA BETWEEN B.InicioPeriodo AND B.FInPeriodo
ORDER BY
    A.IDLegislador, B.InicioPeriodo, C.Fecha;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,IDLegislador,Primer_Periodo,InicioPeriodo,FinPeriodo,Fecha,Periodo
0,A000001,1951-01-03,1951-01-03,1953-01-03,1951-12-31,0
1,A000001,1951-01-03,1951-01-03,1953-01-03,1952-12-31,1
2,A000002,1947-01-03,1947-01-03,1949-01-03,1947-12-31,0
3,A000002,1947-01-03,1947-01-03,1949-01-03,1948-12-31,1
4,A000002,1947-01-03,1949-01-03,1951-01-03,1949-12-31,2
5,A000002,1947-01-03,1949-01-03,1951-01-03,1950-12-31,3
6,A000002,1947-01-03,1951-01-03,1953-01-03,1951-12-31,4
7,A000002,1947-01-03,1951-01-03,1953-01-03,1952-12-31,5
8,A000002,1947-01-03,1953-01-03,1955-01-03,1953-12-31,6
9,A000002,1947-01-03,1953-01-03,1955-01-03,1954-12-31,7


In [23]:
# Con la columna periodo calculada en el punto anterior, calcular el numero de legisladores en cada uno de los periodos */

query="""
SELECT
    Periodo,
    count(distinct IDLegislador) as numero_legisladores
FROM
    (
    SELECT
        A.IDLegislador,
        A.Primer_Periodo,
        B.InicioPeriodo,
        B.FinPeriodo,
        C.Fecha,
        COALESCE(timestampdiff(YEAR,A.primer_periodo,C.fecha),0) as Periodo
    FROM
        (
        SELECT
            IDLegislador,
            MIN(InicioPeriodo) as Primer_Periodo
        FROM
            cohort.legisladoresusa
        GROUP BY 1
        ) AS A 
        INNER JOIN 
        cohort.legisladoresusa AS B ON A.IDLegislador=B.IDLegislador
        LEFT JOIN 
        cohort.tablafechas AS C ON C.FECHA BETWEEN B.InicioPeriodo AND B.FInPeriodo
    ORDER BY A.IDLegislador, B.InicioPeriodo, C.Fecha
    ) AS M
GROUP BY 
    1
ORDER BY 
    1 ASC;
"""

df = pd.read_sql(text(query), engine)
df.head(10)

,Periodo,numero_legisladores
0,0,12518
1,1,12328
2,2,8162
3,3,8065
4,4,5853
5,5,5786
6,6,4360
7,7,4338
8,8,3513
9,9,3476


In [24]:
# De la tabla anterior, calcular una columna con el porcentaje de retencion respecto del periodo 0

query="""
SELECT
    periodo,
    numero_legisladores,
    first_value(numero_legisladores) over (order by periodo) as poblacion_inicial,
    100*(numero_legisladores / first_value(numero_legisladores) over (order by periodo)) as porcentaje_retencion
FROM
    (
    SELECT
        Periodo,
        count(distinct IDLegislador) as numero_legisladores
    FROM
        (
        SELECT
            A.IDLegislador,
            A.Primer_Periodo,
            B.InicioPeriodo,
            B.FinPeriodo,
            C.Fecha,
            COALESCE(timestampdiff(YEAR,A.primer_periodo,C.fecha),0) as Periodo
        FROM
            (
            SELECT
                IDLegislador,
                MIN(InicioPeriodo) as Primer_Periodo
            FROM
                cohort.legisladoresusa
            GROUP BY 1
            ) AS A 
            INNER JOIN 
            cohort.legisladoresusa AS B ON A.IDLegislador=B.IDLegislador
            LEFT JOIN 
            cohort.tablafechas AS C ON C.FECHA BETWEEN B.InicioPeriodo AND B.FInPeriodo
        ORDER BY A.IDLegislador, B.InicioPeriodo, C.Fecha
        ) AS M
    GROUP BY 
        1
    ORDER BY 
        1 ASC
    ) as N
;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,periodo,numero_legisladores,poblacion_inicial,porcentaje_retencion
0,0,12518,12518,100.00
1,1,12328,12518,98.48
2,2,8162,12518,65.20
3,3,8065,12518,64.43
4,4,5853,12518,46.76
5,5,5786,12518,46.22
6,6,4360,12518,34.83
7,7,4338,12518,34.65
8,8,3513,12518,28.06
9,9,3476,12518,27.77


##### 4-5 Cohortes derivadas de la serie de tiempo 

In [26]:
# De la tabla "Supertienda" obtener los clientes y la fecha de su primera orden.

query="""
select
    nombre_del_cliente,
    min(fecha_de_orden) as primera_orden
from
    cohort.supertienda
group by
    1;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,nombre_del_cliente,primera_orden
0,Darren Powers,2019-01-03
1,Phillina Ober,2019-01-04
2,Mick Brown,2019-01-05
3,Maria Etezadi,2019-01-06
4,Jack O'Briant,2019-01-06
5,Lycoris Saunders,2019-01-06
6,Vivek Sundaresam,2019-01-07
7,Melanie Seite,2019-01-09
8,Anthony Jacobs,2019-01-10
9,Seth Vernon,2019-01-11


In [27]:
# Obtener el año de la primera orden y a partir de su primera orden calcular cuantos clientes regresaron 1 mes despues, 2 meses despues, ..., n meses despues

query="""
select
    extract(year from primera_orden) as primer_anio,
    timestampdiff(month, a.primera_orden, b.fecha_de_orden) as periodo,
    count(distinct a.nombre_del_cliente) as clientes_retenidos
from
    (
    select
        nombre_del_cliente,
        min(fecha_de_orden) as primera_orden
    from
        cohort.supertienda
    group by
        1
    ) as a
    inner join 
    cohort.supertienda as b on a.nombre_del_cliente = b.Nombre_del_Cliente
group by 
    1,2;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,primer_anio,periodo,clientes_retenidos
0,2019,0,605
1,2019,1,51
2,2019,2,51
3,2019,3,51
4,2019,4,37
5,2019,5,68
6,2019,6,56
7,2019,7,65
8,2019,8,60
9,2019,9,50


In [28]:
# Usando la consulta anterior, calcular el porcentaje de retención respecto del periodo 0

query="""
select
    primer_anio,
    periodo, 
    clientes_retenidos,
    first_value(clientes_retenidos) over (partition by primer_anio order by periodo) as tamanio_cohorte,
    100 * clientes_retenidos / first_value(clientes_retenidos) over (partition by primer_anio order by periodo) as porcentaje_retencion
from
    (
    select	
        extract(year from primera_orden) as primer_anio,
        timestampdiff(month, a.primera_orden, b.fecha_de_orden) as periodo,
        count(distinct a.nombre_del_cliente) as clientes_retenidos
    from
        (
        select
            nombre_del_cliente,
            min(fecha_de_orden) as primera_orden
        from
            cohort.supertienda
        group by 
            1
        ) as a
        inner join
        cohort.supertienda as b on a.nombre_del_cliente = b.Nombre_del_Cliente
    group by 
        1,2
    ) as b;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,primer_anio,periodo,clientes_retenidos,tamanio_cohorte,porcentaje_retencion
0,2019,0,605,605,100.0000
1,2019,1,51,605,8.4298
2,2019,2,51,605,8.4298
3,2019,3,51,605,8.4298
4,2019,4,37,605,6.1157
5,2019,5,68,605,11.2397
6,2019,6,56,605,9.2562
7,2019,7,65,605,10.7438
8,2019,8,60,605,9.9174
9,2019,9,50,605,8.2645


##### Generar una cohorte con la categoria de la primera compra de los clientes

In [29]:
# Obtener la categoria de la primera compra que hizo cada cliente

query="""
select
    nombre_del_cliente,
    min(fecha_de_orden) as primera_orden,
    first_value(categoria) over (partition by nombre_del_cliente order by fecha_de_orden) as categoria_primera_compra
from
    cohort.supertienda
group by
    1;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,nombre_del_cliente,primera_orden,categoria_primera_compra
0,Aaron Bergman,2019-02-18,Suministros de Oficina
1,Aaron Hawkins,2019-04-22,Suministros de Oficina
2,Aaron Smayling,2019-07-27,Suministros de Oficina
3,Adam Bellavance,2020-09-18,Suministros de Oficina
4,Adam Hart,2019-11-16,Suministros de Oficina
5,Adam Shillingsburg,2019-09-22,Muebles
6,Adrian Barton,2019-12-20,Suministros de Oficina
7,Adrian Hane,2019-07-18,Muebles
8,Adrian Shami,2021-11-13,Muebles
9,Aimee Bixby,2019-03-05,Suministros de Oficina


In [30]:
# Con la consulta anterior, agrupar por categoria_primera_compra y periodo de compra, y obtener el total de clientes retenidos */

query="""
select
    categoria_primera_compra,
    timestampdiff(month, a.primera_orden, b.fecha_de_orden) as periodo,
    count(distinct a.nombre_del_cliente) as clientes_retenidos
from
    (
    select
        nombre_del_cliente,
        min(fecha_de_orden) as primera_orden,
			     first_value(categoria) over (partition by nombre_deL_cliente order by fecha_de_orden) as categoria_primera_compra
    from
        cohort.supertienda
    group by
        1
    ) as a
    INNER JOIN 
    cohort.supertienda as b on a.nombre_del_cliente = b.nombre_del_cliente
group by 
    1,2;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,categoria_primera_compra,periodo,clientes_retenidos
0,Muebles,0,285
1,Muebles,1,23
2,Muebles,2,22
3,Muebles,3,25
4,Muebles,4,22
5,Muebles,5,27
6,Muebles,6,24
7,Muebles,7,31
8,Muebles,8,31
9,Muebles,9,28


In [31]:
# Con la consulta anterior obtener el porcentaje de retencion respecto del periodo 0

query="""
select
    categoria_primera_compra,
    periodo,
    clientes_retenidos,
    first_value(clientes_retenidos) over (partition by categoria_primera_compra order by periodo) as tamanio_cohorte,
    100 * clientes_retenidos / first_value(clientes_retenidos) over (partition by categoria_primera_compra order by periodo) as porcentaje_retencion
from
    (
    select
        categoria_primera_compra,
        timestampdiff(month, a.primera_orden, b.fecha_de_orden) as periodo,
        count(distinct a.nombre_del_cliente) as clientes_retenidos
    from
        (
        select
            nombre_del_cliente,
            min(fecha_de_orden) as primera_orden,
            first_value(categoria) over (partition by nombre_del_cliente order by fecha_de_orden) as categoria_primera_compra
        from
            cohort.supertienda
        group by
            1
        ) as a
        inner join 
        cohort.supertienda as b on a.nombre_del_cliente=b.nombre_del_cliente
    group by
        1,2
    ) as c
group by 1,2;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,categoria_primera_compra,periodo,clientes_retenidos,tamanio_cohorte,porcentaje_retencion
0,Muebles,0,285,285,100.0000
1,Muebles,1,23,285,8.0702
2,Muebles,2,22,285,7.7193
3,Muebles,3,25,285,8.7719
4,Muebles,4,22,285,7.7193
5,Muebles,5,27,285,9.4737
6,Muebles,6,24,285,8.4211
7,Muebles,7,31,285,10.8772
8,Muebles,8,31,285,10.8772
9,Muebles,9,28,285,9.8246


##### 4-6 Calculo en Fechas Diferentes

##### En el capitulo anterior se vio el caso de una cohorte definida por la fecha de compra del primer producto. Ahora se definirá la cohorte a partir de una fecha fija que sea de interés para un estudio

In [32]:
# Vista previa de la tabla legisladoresusa

query="""
select
    *
from
    cohort.legisladoresusa 
limit
    5;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,IDLegislador,DipSen,InicioPeriodo,FinPeriodo,Estado
0,B000944,rep,1993-01-05,1995-01-03,OH
1,C000127,rep,1993-01-05,1995-01-03,WA
2,C000141,rep,1987-01-06,1989-01-03,MD
3,C000174,rep,1983-01-03,1985-01-03,DE
4,C001070,sen,2007-01-04,2013-01-03,PA


In [33]:
# Obtener los diputados o senadores que estuvieron activos en algún momento del año 2000

query="""
select
    IDLegislador,
    DipSen,
    date("2000-01-01") as inicio_cohorte,
    min(InicioPeriodo) as inicio_periodo
from
    cohort.legisladoresusa
where
    InicioPeriodo<="2000-12-31" and FinPeriodo>="2000-01-01"
group by
    1,2,3;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,IDLegislador,DipSen,inicio_cohorte,inicio_periodo
0,C001035,sen,2000-01-01,1997-01-07
1,E000285,sen,2000-01-01,1997-01-07
2,U000039,rep,2000-01-01,1999-01-06
3,B001230,rep,2000-01-01,1999-01-06
4,L000557,rep,2000-01-01,1999-01-06
5,N000179,rep,2000-01-01,1999-01-06
6,S001145,rep,2000-01-01,1999-01-06
7,S001148,rep,2000-01-01,1999-01-06
8,T000460,rep,2000-01-01,1999-01-06
9,T000461,rep,2000-01-01,1999-01-06


In [34]:
# Agregar los datos de la consulta anterior a la tabla legisladoresusa

query="""
SELECT
    *
FROM
    (
    select
        IDLegislador,
        DipSen,
        date("2000-01-01") as inicio_cohorte,
        min(InicioPeriodo) as inicio_periodo
    from
        cohort.legisladoresusa
    where
        InicioPeriodo<="2000-12-31" and FinPeriodo>="2000-01-01"
    group by
        1,2,3
    ) as a
    inner join
    cohort.legisladoresusa as b on a.IDLegislador=b.IDLegislador and b.InicioPeriodo >= a.inicio_periodo;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,IDLegislador,DipSen,inicio_cohorte,inicio_periodo,IDLegislador,DipSen,InicioPeriodo,FinPeriodo,Estado
0,C001035,sen,2000-01-01,1997-01-07,C001035,sen,1997-01-07,2003-01-03,ME
1,E000285,sen,2000-01-01,1997-01-07,E000285,sen,1997-01-07,2003-01-03,WY
2,U000039,rep,2000-01-01,1999-01-06,U000039,rep,1999-01-06,2001-01-03,NM
3,B001230,rep,2000-01-01,1999-01-06,B001230,rep,1999-01-06,2001-01-03,WI
4,L000557,rep,2000-01-01,1999-01-06,L000557,rep,1999-01-06,2001-01-03,CT
5,N000179,rep,2000-01-01,1999-01-06,N000179,rep,1999-01-06,2001-01-03,CA
6,S001145,rep,2000-01-01,1999-01-06,S001145,rep,1999-01-06,2001-01-03,IL
7,S001148,rep,2000-01-01,1999-01-06,S001148,rep,1999-01-06,2001-01-03,ID
8,T000460,rep,2000-01-01,1999-01-06,T000460,rep,1999-01-06,2001-01-03,CA
9,T000461,rep,2000-01-01,1999-01-06,T000461,rep,1999-01-06,2001-01-03,PA


In [36]:
# Con la consulta anterior rellenar las fechas intermedias entre inicio periodo y fin periodo usando la tabla auxiliar "tablafechas"

query="""
select
    *,
    timestampdiff(year,a.inicio_cohorte,c.fecha) as periodo
from
    (
    select
        IDLegislador,
        DipSen,
        date("2000-01-01") as inicio_cohorte,
        min(InicioPeriodo) as inicio_periodo
    from
        cohort.legisladoresusa
    where
        InicioPeriodo<="2000-12-31" and FinPeriodo>="2000-01-01"
    group by
        1,2,3
    ) as a
    inner join
    cohort.legisladoresusa as b on a.IDLegislador=b.IDLegislador and b.InicioPeriodo >= a.inicio_periodo
    left join
    cohort.tablafechas as c on c.fecha between b.InicioPeriodo and b.FinPeriodo and c.year >= 2000   /*Crear una fila para cada fecha en la tabla fechas si la fecha esta entre el inicio_periodo y fin_periodo del legislador, conservar solo del 2000 en adelante.*/
where
    timestampdiff(year,a.inicio_cohorte,c.fecha) IS NOT NULL;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,IDLegislador,DipSen,inicio_cohorte,inicio_periodo,IDLegislador,DipSen,InicioPeriodo,FinPeriodo,Estado,Year,Mes,Dia,Dias en Year,Fecha,periodo
0,C001035,sen,2000-01-01,1997-01-07,C001035,sen,1997-01-07,2003-01-03,ME,2000,12,31,366,2000-12-31,0
1,C001035,sen,2000-01-01,1997-01-07,C001035,sen,1997-01-07,2003-01-03,ME,2001,12,31,365,2001-12-31,1
2,C001035,sen,2000-01-01,1997-01-07,C001035,sen,1997-01-07,2003-01-03,ME,2002,12,31,365,2002-12-31,2
3,E000285,sen,2000-01-01,1997-01-07,E000285,sen,1997-01-07,2003-01-03,WY,2000,12,31,366,2000-12-31,0
4,E000285,sen,2000-01-01,1997-01-07,E000285,sen,1997-01-07,2003-01-03,WY,2001,12,31,365,2001-12-31,1
5,E000285,sen,2000-01-01,1997-01-07,E000285,sen,1997-01-07,2003-01-03,WY,2002,12,31,365,2002-12-31,2
6,U000039,rep,2000-01-01,1999-01-06,U000039,rep,1999-01-06,2001-01-03,NM,2000,12,31,366,2000-12-31,0
7,B001230,rep,2000-01-01,1999-01-06,B001230,rep,1999-01-06,2001-01-03,WI,2000,12,31,366,2000-12-31,0
8,L000557,rep,2000-01-01,1999-01-06,L000557,rep,1999-01-06,2001-01-03,CT,2000,12,31,366,2000-12-31,0
9,N000179,rep,2000-01-01,1999-01-06,N000179,rep,1999-01-06,2001-01-03,CA,2000,12,31,366,2000-12-31,0


In [38]:
# Con la consulta anterior calcular el periodo en años a partir del año 2000 y el numero de legisladores en cada periodo

query = """
select
    a.dipsen,
    timestampdiff(year,a.inicio_cohorte,c.fecha) as periodo,
    count(distinct a.IDLegislador) as cohorte_retenida
from
    (
    select
        IDLegislador,
        DipSen,
        date("2000-01-01") as inicio_cohorte,
        min(InicioPeriodo) as inicio_periodo
    from
        cohort.legisladoresusa
    where
        InicioPeriodo<="2000-12-31" and FinPeriodo>="2000-01-01"
    group by 1,2,3
    ) as a
    inner join
    cohort.legisladoresusa as b on a.IDLegislador=b.IDLegislador and b.InicioPeriodo >= a.inicio_periodo
    left join 
    cohort.tablafechas as c on c.fecha between b.InicioPeriodo and b.FinPeriodo and c.year >= 2000
where timestampdiff(year,a.inicio_cohorte,c.fecha) IS NOT NULL
group by 1,2;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,dipsen,periodo,cohorte_retenida
0,rep,0,439
1,rep,1,392
2,rep,2,389
3,rep,3,340
4,rep,4,338
5,rep,5,308
6,rep,6,306
7,rep,7,267
8,rep,8,262
9,rep,9,223


In [40]:
# Con la consulta anterior calcular el porcentaje respecto del año de inicio de la cohorte */

query="""
select
    dipsen,
    periodo,
    cohorte_retenida,
    first_value(cohorte_retenida) over (partition by dipsen order by periodo) as cohorte_inicial,
    100 * cohorte_retenida / first_value(cohorte_retenida) over (partition by dipsen order by periodo) as porcentaje_retencion
from
    (
		  select
        a.dipsen,
        coalesce(timestampdiff(year,a.inicio_cohorte,c.fecha),0) as periodo,
        count(distinct a.IDLegislador) as cohorte_retenida
    from
        (
        select
            IDLegislador,
            DipSen,
            date("2000-01-01") as inicio_cohorte,
            min(InicioPeriodo) as inicio_periodo
        from
            cohort.legisladoresusa
        where
            InicioPeriodo<="2000-12-31" and FinPeriodo>="2000-01-01"
        group by
            1,2,3
        ) as a
        inner join
        cohort.legisladoresusa as b on a.IDLegislador=b.IDLegislador and b.InicioPeriodo >= a.inicio_periodo
        left join
        cohort.tablafechas as c on c.fecha between b.InicioPeriodo and b.FinPeriodo and c.year >= 2000
    group by 1,2
    ) as b;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,dipsen,periodo,cohorte_retenida,cohorte_inicial,porcentaje_retencion
0,rep,0,440,440,100.0000
1,rep,1,392,440,89.0909
2,rep,2,389,440,88.4091
3,rep,3,340,440,77.2727
4,rep,4,338,440,76.8182
5,rep,5,308,440,70.0000
6,rep,6,306,440,69.5455
7,rep,7,267,440,60.6818
8,rep,8,262,440,59.5455
9,rep,9,223,440,50.6818


##### Analisis de Supervivencia

In [47]:
# Por legislador, obtener el siglo en que inicio su periodo, la fecha de inicio de su primer periodo, la fecha de inicio de su ultimo periodo y años de permanencia */

query="""
SELECT
    idlegislador,
    FLOOR((year(min(inicioperiodo))+99)/100) as siglo,
    min(inicioperiodo) as primer_periodo,
    max(inicioperiodo) as ultimo_periodo,
    timestampdiff(year,min(inicioperiodo),max(inicioperiodo)) as permanencia
FROM
    cohort.legisladoresusa
GROUP BY
    1
ORDER BY
    permanencia;
"""
df = pd.read_sql(text(query), engine)
df.head(10)


,idlegislador,siglo,primer_periodo,ultimo_periodo,permanencia
0,C000555,20,1985-01-03,1985-01-03,0
1,E000033,20,1985-01-03,1985-01-03,0
2,G000499,20,1985-01-03,1985-01-03,0
3,L000411,20,1985-01-03,1985-01-03,0
4,M000860,20,1985-01-03,1985-01-03,0
5,K000309,20,1987-01-06,1987-01-06,0
6,S000991,20,1985-01-03,1985-01-03,0
7,W000038,20,1985-01-03,1985-01-03,0
8,S000055,20,1986-11-05,1987-01-06,0
9,D000106,20,1987-01-06,1987-01-06,0


In [49]:
# Con la consulta anterior, agrupar por siglo y obtener el numero y porcentaje de legisladores que estuvieron 10 o más años

query="""
SELECT
    siglo,
    COUNT(distinct idlegislador) AS cohorte_inicial,
    COUNT(distinct case when permanencia>=10 then idlegislador end) AS mas_de_10_anios,
    COUNT(distinct case when permanencia>=10 then idlegislador end) / COUNT(distinct idlegislador) AS porcentaje
FROM
    (
    SELECT
        idlegislador,
        FLOOR((year(min(inicioperiodo))+99)/100) as siglo,
        min(inicioperiodo) as primer_periodo,
        max(inicioperiodo) as ultimo_periodo,
        timestampdiff(year,min(inicioperiodo),max(inicioperiodo)) as permanencia
    FROM
        cohort.legisladoresusa
    GROUP BY
        1
    ORDER BY
        permanencia
    ) AS A
GROUP BY 1;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,siglo,cohorte_inicial,mas_de_10_anios,porcentaje
0,18,368,83,0.2255
1,19,6299,892,0.1416
2,20,5091,1853,0.3640
3,21,760,119,0.1566


##### Analisis de retorno de clientes

In [ ]:
# Desglosado por año, obtener cuantos de los clientes hicieron su primera compra en articulos de oficina 

query="""
select
    YEAR(primer_orden) as cohort_anual,
    count(distinct nombre_del_cliente) as oficina
from
    (
    select
        nombre_del_cliente,
        min(fecha_de_orden) as primer_orden
    from
        cohort.supertienda
    where
        categoria = "Suministros de oficina"
    group by 1
    ) as A
group by 1;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,cohort_anual,oficina
0,2019,518
1,2020,169
2,2021,83
3,2022,23


In [ ]:
# Con la consulta anterior, obtener desglosado por año, el numero de clientes cuya primera compra fue en articulos de oficina 
# y posteriormente vuelven para comprar articulos de tecnologia

query="""
SELECT
    year(a.primer_orden) as anio_cohorte,
    count(distinct b.nombre_del_cliente) as oficina_tecnologia
FROM
    (
    SELECT
        nombre_del_cliente,
        min(fecha_de_orden) as primer_orden
    FROM
        cohort.supertienda
    WHERE
        categoria = "Suministros de oficina"
    GROUP BY
        1
    ) as a
    INNER JOIN
    cohort.supertienda as b on a.nombre_del_cliente = b.nombre_del_cliente
WHERE b.Categoria="Tecnología" AND b.Fecha_de_Orden > a.primer_orden
GROUP BY 1;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,anio_cohorte,oficina_tecnologia
0,2019,424
1,2020,128
2,2021,53
3,2022,8


In [63]:
# Con la consulta anterior obtener el porcentaje de clientes que compraron primero articulos de oficina y regresan en un tiempo futuro por articulos de tecnologia */

query="""
SELECT
    AA.anio_cohorte,
    AA.oficina,
    BB.oficina_tecnologia,
    BB.oficina_tecnologia / AA.oficina AS porcentaje	
FROM
    (
    SELECT
        year(a.primer_orden) as anio_cohorte,
			     count(distinct a.nombre_del_cliente) as oficina
    FROM
        (
        SELECT
            nombre_del_cliente,
            min(fecha_de_orden) as primer_orden
        FROM
            cohort.supertienda
        WHERE
            categoria = "Suministros de oficina"
        group by 1
        ) as a
    GROUP BY 1
    ) AS AA
    LEFT JOIN
    (
    SELECT
        year(a.primer_orden) as anio_cohorte,
        count(distinct b.nombre_del_cliente) as oficina_tecnologia
    FROM
        (
        SELECT
            nombre_del_cliente,
            min(fecha_de_orden) as primer_orden
        FROM
            cohort.supertienda
        WHERE
            categoria = "Suministros de oficina"
        GROUP BY 1
        ) as a
        INNER JOIN
        cohort.supertienda as b on a.nombre_del_cliente = b.nombre_del_cliente
    WHERE b.Categoria="Tecnología" AND b.Fecha_de_Orden > a.primer_orden
    GROUP BY 1
    ) AS BB ON AA.anio_cohorte = BB.anio_cohorte;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,anio_cohorte,oficina,oficina_tecnologia,porcentaje
0,2019,518,424,0.8185
1,2020,169,128,0.7574
2,2021,83,53,0.6386
3,2022,23,8,0.3478


##### En los ejemplos anteriores solo se pidio que el cliente regresara pero no se especifica en cuantos periodos deberia de regresar

In [65]:
# Con la consulta anterior obtener el porcentaje de clientes que compraron primero articulos de oficina y en 12 meses o menos regresan por articulos de tecnologia

query="""
SELECT
    AA.anio_cohorte,
    AA.oficina,
    BB.oficina_tecnologia,
    BB.oficina_tecnologia / AA.oficina AS porcentaje
FROM
    (
    SELECT 
        year(a.primer_orden) as anio_cohorte,
        count(distinct a.nombre_del_cliente) as oficina
    FROM
        (
        SELECT
            nombre_del_cliente,
				        min(fecha_de_orden) as primer_orden
        FROM
            cohort.supertienda
        where
            categoria = "Suministros de oficina"
        group by
            1
        ) as a
    GROUP BY 1
    ) AS AA
    LEFT JOIN
    (
    SELECT
        year(a.primer_orden) as anio_cohorte,
        count(distinct b.nombre_del_cliente) as oficina_tecnologia
    FROM
        (
        SELECT
            nombre_del_cliente,
            min(fecha_de_orden) as primer_orden
        FROM
            cohort.supertienda
        where
            categoria = "Suministros de oficina"
        group by 1
        ) as a
        INNER JOIN
        cohort.supertienda as b on a.nombre_del_cliente = b.nombre_del_cliente
    WHERE b.Categoria="Tecnología" AND b.Fecha_de_Orden > a.primer_orden AND timestampdiff(MONTH,a.primer_orden,b.fecha_de_orden) <= 12
    GROUP BY 1
    ) AS BB 
    ON AA.anio_cohorte = BB.anio_cohorte;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,anio_cohorte,oficina,oficina_tecnologia,porcentaje
0,2019,518,166,0.3205
1,2020,169,69,0.4083
2,2021,83,40,0.4819
3,2022,23,8,0.3478


##### Calculos Acumulados

In [70]:
# Desglosado por cliente, obtener la categoria de su primera orden, fecha de su primera orden y obtener la fecha 10 meses despues de su primera orden */

query="""
select
    distinct nombre_del_cliente,
    first_value(categoria) over (partition by nombre_del_cliente order by fecha_de_orden asc) as primera_categoria,
    min(fecha_de_orden) over (partition by nombre_del_cliente) as primera_orden,
    min(fecha_de_orden) over (partition by nombre_del_cliente) + interval 10 month as primeros_10_meses
from
    cohort.supertienda;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,nombre_del_cliente,primera_categoria,primera_orden,primeros_10_meses
0,Aaron Bergman,Suministros de Oficina,2019-02-18,2019-12-18
1,Aaron Hawkins,Suministros de Oficina,2019-04-22,2020-02-22
2,Aaron Smayling,Suministros de Oficina,2019-07-27,2020-05-27
3,Adam Bellavance,Suministros de Oficina,2020-09-18,2021-07-18
4,Adam Hart,Suministros de Oficina,2019-11-16,2020-09-16
5,Adam Shillingsburg,Muebles,2019-09-22,2020-07-22
6,Adrian Barton,Suministros de Oficina,2019-12-20,2020-10-20
7,Adrian Hane,Muebles,2019-07-18,2020-05-18
8,Adrian Shami,Muebles,2021-11-13,2022-09-13
9,Aimee Bixby,Tecnología,2019-03-05,2020-01-05


In [77]:
# Desglosado por año y primera categoria de compra, calcular el numero de compradores inicial, el numero de ordenes, 
# las ordenes promedio por cliente, la suma de ventas y venta promedio */

query="""
select
    extract(year from a.primera_orden) as anio,
    primera_categoria,
    count(distinct a.nombre_del_cliente) as poblacion_inicial,
    count(b.fecha_de_orden) as ordenes,
    count(b.fecha_de_orden) / count(distinct a.nombre_del_cliente) as ordenes_por_cliente,
    round(sum(b.ventas),2) as ventas_acumuladas,
    round(sum(b.ventas) / count(distinct a.nombre_del_cliente),2) as venta_promedio
from
    (
    select
        distinct nombre_del_cliente,
        first_value(categoria) over (partition by nombre_del_cliente order by fecha_de_orden asc) as primera_categoria,
        min(fecha_de_orden) over (partition by nombre_del_cliente) as primera_orden,
        min(fecha_de_orden) over (partition by nombre_del_cliente) + interval 10 month as primeros_10_meses
    from 	cohort.supertienda
    ) as a
    LEFT JOIN
    cohort.supertienda as b ON a.nombre_del_cliente = b.nombre_del_cliente AND b.fecha_de_orden between a.primera_orden and a.primeros_10_meses
group by 1,2;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,anio,primera_categoria,poblacion_inicial,ordenes,ordenes_por_cliente,ventas_acumuladas,venta_promedio
0,2019,Muebles,125,506,4.0480,157126.21,1257.01
1,2019,Suministros de Oficina,379,1491,3.9340,300034.36,791.65
2,2019,Tecnología,101,440,4.3564,112910.93,1117.93
3,2020,Muebles,25,144,5.7600,32500.72,1300.03
4,2020,Suministros de Oficina,81,306,3.7778,63755.42,787.10
5,2020,Tecnología,25,95,3.8000,23223.95,928.96
6,2021,Muebles,16,63,3.9375,16438.71,1027.42
7,2021,Suministros de Oficina,29,148,5.1034,27017.43,931.64
8,2021,Tecnología,8,53,6.6250,22438.04,2804.75
9,2022,Muebles,4,25,6.2500,3227.96,806.99


##### 4-10 Analisis Transversal mediante Cohorte

In [80]:
# Obtener el numero de legisladores por año

query = """
select
    b.fecha,
    count(distinct a.idlegislador) as legisladores
from
    cohort.legisladoresusa as a
    inner join
    cohort.tablafechas as b on b.Fecha between a.InicioPeriodo and a.FinPeriodo
group by
    1;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,fecha,legisladores
0,1789-12-31,89
1,1790-12-31,95
2,1791-12-31,99
3,1792-12-31,101
4,1793-12-31,141
5,1794-12-31,140
6,1795-12-31,145
7,1796-12-31,150
8,1797-12-31,152
9,1798-12-31,155


In [ ]:
# Obtener el id del legislador, la fecha de su primer periodo y siglo de su primer periodo */

query = """
select
    IDLegislador,
    min(InicioPeriodo) as primer_periodo,
    floor((year(min(InicioPeriodo))+99)/100) as siglo_primer_periodo
from
    cohort.legisladoresusa
group by
    1;
"""
df = pd.read_sql(text(query), engine)
df.head(10)

,IDLegislador,primer_periodo,siglo_primer_periodo
0,B000944,1993-01-05,20
1,C000127,1993-01-05,20
2,C000141,1987-01-06,20
3,C000174,1983-01-03,20
4,C001070,2007-01-04,21
5,F000062,1992-11-10,20
6,F000469,2019-01-03,21
7,K000367,2007-01-04,21
8,M000639,1993-01-05,20
9,S000033,1991-01-03,20
